# Heart Disease Prediction Using Ridge- and Lasso-Regularized Logistic Regression

### Machine Learning Project

**Goal:** Build and compare three binary classification models for heart disease prediction:

1. Logistic Regression
2. Ridge-regularized Logistic Regression (L2)
3. Lasso-regularized Logistic Regression (L1)

The notebook follows a complete machine-learning workflow:

**Dataset → EDA → Preprocessing → Train/Test Split → Model Training → Evaluation → Model Comparison → Best Model → Final Prediction**

> **Note:** This project is for academic/educational purposes and is not a medical diagnosis system.


## 1. Import Libraries and Load the Dataset

We use the **UCI Heart Disease dataset** through `ucimlrepo`.

The dataset contains patient health attributes and a Heart_Disease called `num`.  
For this project, `num` is converted into a binary Heart_Disease:

- `0` → No heart disease
- `1` → Heart disease present


In [ ]:
!pip -q install ucimlrepo

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

# Load UCI Heart Disease dataset
heart_disease = fetch_ucirepo(id=45)

X_raw = heart_disease.data.features.copy()
y_raw = heart_disease.data.targets.copy()

df = pd.concat([X_raw, y_raw], axis=1)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Give the dataset meaningful column names
df = df.rename(columns={
    "age": "Age", "sex": "Sex", "cp": "Chest_Pain_Type",
    "trestbps": "Resting_Blood_Pressure", "chol": "Cholesterol",
    "fbs": "Fasting_Blood_Sugar", "restecg": "Resting_ECG",
    "thalach": "Maximum_Heart_Rate", "exang": "Exercise_Induced_Angina",
    "oldpeak": "ST_Depression", "slope": "ST_Segment_Slope",
    "ca": "Major_Vessels", "thal": "Thalassemia", "num": "Heart_Disease"
})

# Convert 0 = no disease and 1-4 = disease into 0/1
df["Heart_Disease"] = (df["Heart_Disease"] > 0).astype(int)

print("Meaningful column names used:")
print(df.columns.tolist())
df.head()


## 2. Understand the Dataset

Before training a model, we inspect the columns, data types, summary statistics, missing values, and duplicate records.

The original Heart_Disease `num` has values from 0 to 4. For binary classification, all values greater than 0 are treated as disease present.


In [ ]:
print("Descriptive statistics:")
display(df.describe().T.round(2))

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nDuplicate rows:", df.duplicated().sum())


# 3. Exploratory Data Analysis (EDA)

EDA helps us understand the dataset before model training.

We will examine:

- Target distribution
- Age distribution
- Age versus disease
- Sex distribution
- Chest-pain type versus disease
- Cholesterol and maximum heart rate
- Resting blood pressure
- Feature correlations

The purpose of EDA is **understanding the data**, not proving that one feature alone causes heart disease.


In [ ]:
# Target distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Heart_Disease")
plt.title("Heart Disease Distribution")
plt.xlabel("Target (0 = No Disease, 1 = Disease)")
plt.ylabel("Number of Patients")
plt.show()


In [ ]:
# Age distribution and Age versus Heart_Disease
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["Age"], bins=15, kde=True, ax=axes[0])
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Number of Patients")

sns.boxplot(data=df, x="Heart_Disease", y="Age", ax=axes[1])
axes[1].set_title("Age vs Heart Disease")
axes[1].set_xlabel("Target")
axes[1].set_ylabel("Age")

plt.tight_layout()
plt.show()


In [ ]:
# Sex distribution and Sex versus Heart_Disease
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="Sex", ax=axes[0])
axes[0].set_title("Sex Distribution")
axes[0].set_xlabel("Sex (0 = Female, 1 = Male)")
axes[0].set_ylabel("Number of Patients")

sns.countplot(data=df, x="Sex", hue="Heart_Disease", ax=axes[1])
axes[1].set_title("Sex vs Heart Disease")
axes[1].set_xlabel("Sex")
axes[1].set_ylabel("Number of Patients")

plt.tight_layout()
plt.show()


In [ ]:
# Chest pain type versus Heart_Disease
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Chest_Pain_Type", hue="Heart_Disease")
plt.title("Chest Pain Type vs Heart Disease")
plt.xlabel("Chest Pain Type")
plt.ylabel("Number of Patients")
plt.show()


In [ ]:
# Important numerical feature distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df["Cholesterol"], bins=20, kde=True, ax=axes[0])
axes[0].set_title("Cholesterol Distribution")
axes[0].set_xlabel("Cholesterol")

sns.boxplot(data=df, x="Heart_Disease", y="Cholesterol", ax=axes[1])
axes[1].set_title("Cholesterol vs Heart Disease")
axes[1].set_xlabel("Target")
axes[1].set_ylabel("Cholesterol")

sns.boxplot(data=df, x="Heart_Disease", y="Maximum_Heart_Rate", ax=axes[2])
axes[2].set_title("Maximum Heart Rate vs Heart Disease")
axes[2].set_xlabel("Target")
axes[2].set_ylabel("Maximum Heart Rate")

plt.tight_layout()
plt.show()


In [ ]:
# Resting blood pressure
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["Resting_Blood_Pressure"], bins=20, kde=True, ax=axes[0])
axes[0].set_title("Resting Blood Pressure Distribution")
axes[0].set_xlabel("Resting Blood Pressure")
axes[0].set_ylabel("Number of Patients")

sns.boxplot(data=df, x="Heart_Disease", y="Resting_Blood_Pressure", ax=axes[1])
axes[1].set_title("Resting Blood Pressure vs Heart Disease")
axes[1].set_xlabel("Target")
axes[1].set_ylabel("Resting Blood Pressure")

plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr = df.corr(numeric_only=True)

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.3
)

plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


### EDA Summary

From the EDA, we have:

- A binary Heart_Disease suitable for classification.
- Missing values in some medical attributes, which need to be handled before training.
- Different feature scales, so standardization is useful for Logistic Regression.
- Several variables show visible differences across the two Heart_Disease classes.
- The correlation heatmap helps us understand relationships between variables.

EDA is now complete. We move to **model preparation and training**.


# 4. Prepare Data for Machine Learning

We separate:

- `X` = input features
- `y` = Heart_Disease

Then we split the dataset into training and testing sets.

To avoid data leakage, imputation and scaling are placed **inside the model pipeline**. This means they are learned only from the training data.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

X = df.drop("Heart_Disease", axis=1)
y = df["Heart_Disease"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])
print("Number of features:", X_train.shape[1])


# 5. Build the Three Models

### Logistic Regression
This is the **baseline model without regularization**.

### Ridge Logistic Regression
Uses **L2 regularization**. It penalizes large coefficients and can help control overfitting.

### Lasso Logistic Regression
Uses **L1 regularization**. It can shrink some coefficients toward zero and therefore can support feature selection.

For a fair comparison, all three models use the same imputation and scaling steps.

**Important implementation detail:** Scikit-learn's LogisticRegression uses L2 regularization by default. Therefore, the baseline model is explicitly configured with `penalty=None`; otherwise the baseline and Ridge models would be effectively identical.


In [ ]:
def make_model(penalty, solver, C=1.0):
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]

    if penalty is None:
        # Unregularized Logistic Regression baseline
        model = LogisticRegression(
            penalty=None,
            solver=solver,
            max_iter=2000,
            random_state=42
        )
    else:
        model = LogisticRegression(
            penalty=penalty,
            C=C,
            solver=solver,
            max_iter=2000,
            random_state=42
        )

    steps.append(("model", model))
    return Pipeline(steps)

# Baseline: no regularization
logistic = make_model(None, "lbfgs")

# Ridge: L2 regularization
ridge = make_model("l2", "lbfgs", C=1.0)

# Lasso: L1 regularization
lasso = make_model("l1", "liblinear", C=1.0)

models = {
    "Logistic Regression": logistic,
    "Ridge Logistic Regression": ridge,
    "Lasso Logistic Regression": lasso
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: training completed")


# 6. Evaluate the Models

We use:

- **Accuracy** – overall percentage of correct predictions.
- **Precision** – how many predicted positive cases were actually positive.
- **Recall** – how many actual positive cases were detected.
- **F1-score** – balance between precision and recall.
- **ROC-AUC** – ability to distinguish between the two classes across probability thresholds.

These metrics give a more complete view than accuracy alone.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).set_index("Model")

print("Model Performance:")
display(results_df.round(4))

print("\nModel Performance (%):")
display((results_df * 100).round(2))


# 7. Compare the Models

The comparison table is one of the main final outputs of the project.

For this project, we use **ROC-AUC as the primary comparison criterion** because it evaluates how well the model separates the two classes across different thresholds.

The model with the highest ROC-AUC in the test set is selected as the final model.


In [ ]:
best_model_name = results_df["ROC-AUC"].idxmax()
best_model = models[best_model_name]

print("Selected Model:", best_model_name)
print("Reason: Highest test ROC-AUC =", round(results_df.loc[best_model_name, "ROC-AUC"], 4))


In [ ]:
# Visual comparison of evaluation metrics
results_df.plot(kind="bar", figsize=(12, 6))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


# 8. Confusion Matrices

A confusion matrix shows:

- True Positive (TP)
- True Negative (TN)
- False Positive (FP)
- False Negative (FN)

This helps us understand the types of classification errors made by each model.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test,
        y_test,
        display_labels=["No Disease", "Disease"],
        cmap="Blues",
        ax=ax
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()


# 9. ROC Curves

The ROC curve shows the relationship between:

- True Positive Rate (Sensitivity)
- False Positive Rate

A larger area under the ROC curve indicates better class separation.


In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(9, 6))

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 10. Feature Coefficients

Logistic Regression models use coefficients to represent the direction and strength of the relationship between standardized input features and the predicted class.

- Positive coefficient → increases the model's tendency toward the positive class.
- Negative coefficient → decreases the model's tendency toward the positive class.
- Larger absolute coefficient → stronger influence within this model.

This is model interpretation, not proof of medical causation.


In [ ]:
# Extract coefficients from the selected model
coefficients = best_model.named_steps["model"].coef_[0]

feature_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefficients,
    "Absolute Coefficient": np.abs(coefficients)
}).sort_values("Absolute Coefficient", ascending=False)

display(feature_df.round(4))

plt.figure(figsize=(9, 6))
sns.barplot(
    data=feature_df.sort_values("Coefficient"),
    x="Coefficient",
    y="Feature"
)
plt.title(f"Feature Coefficients - {best_model_name}")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


# 11. Final Prediction Demonstration

The final implementation should demonstrate that the trained model can accept a patient's feature values and return:

- Predicted class
- Probability of heart disease

For demonstration, we use one record from the test set. This shows how the trained model can make a prediction on previously unseen data.


In [ ]:
# Use one unseen test sample for a final demonstration
sample = X_test.iloc[[0]]
actual = y_test.iloc[0]

predicted_class = int(best_model.predict(sample)[0])
predicted_probability = best_model.predict_proba(sample)[0, 1]

print("Final Prediction Demonstration")
print("-" * 40)
print("Selected model:", best_model_name)
print("Actual class:", "Heart Disease" if actual == 1 else "No Heart Disease")
print("Predicted class:", "Heart Disease" if predicted_class == 1 else "No Heart Disease")
print(f"Predicted probability of heart disease: {predicted_probability * 100:.2f}%")

print("\nPatient input:")
display(sample)


# 12. Final Results and Conclusion

### What this project accomplished

1. Loaded and inspected the UCI Heart Disease dataset.
2. Converted the original multi-value Heart_Disease into a binary classification Heart_Disease.
3. Performed exploratory data analysis using statistical summaries and visualizations.
4. Identified missing values and handled them using median imputation.
5. Standardized features before Logistic Regression training.
6. Trained three models:
   - Logistic Regression
   - Ridge Logistic Regression
   - Lasso Logistic Regression
7. Compared the models using Accuracy, Precision, Recall, F1-score, and ROC-AUC.
8. Evaluated the models using confusion matrices and ROC curves.
9. Selected the model with the highest test ROC-AUC.
10. Demonstrated a final prediction on an unseen test record.

### Final interpretation

The notebook demonstrates a complete machine-learning classification workflow from raw healthcare data to model evaluation and prediction.

> **Important:** The prediction is an educational ML output and should not be treated as a clinical diagnosis.


# 13. Optional: Save the Selected Model

The following cell saves the selected trained pipeline using Joblib.

Because the pipeline contains the imputer, scaler, and model together, the same preprocessing steps are automatically applied when the saved model is loaded later.


In [ ]:
import joblib

model_filename = "heart_disease_best_model.joblib"
joblib.dump(best_model, model_filename)

print(f"Saved selected model as: {model_filename}")
